# Lab 6 — Annotating images for object detection

**Module:** 7144COMP — Deep Learning Concepts and Techniques  
**Week:** 6  
**Estimated time:** 180 minutes

---

## Learning outcomes

By the end of this lab you should be able to:

1. Explain the difference between **image classification** (what's in this image?) and **object detection** (what's in this image, where, and how many of each?).
2. Read, write, and *validate* labels in the **YOLO text format**, including normalised centre-based bounding box coordinates.
3. Annotate a folder of images using the module's built-in bounding-box annotator and produce a clean YOLO-format label set.
4. Build a valid Ultralytics `data.yaml` specification linking your image folder, label folder, class list, and train/val/test split.
5. Visualise existing annotations by rendering bounding boxes back onto their source images — the critical 'closing-the-loop' step that catches off-by-one errors and class-mapping mistakes.
6. Apply common annotation hygiene rules — tight boxes, consistent class assignment, handling of edge cases like truncation, occlusion, and ambiguous identification.

## Prerequisites

- **Labs 1–5** completed. The pipeline knowledge from Lab 5 will be directly extended here.
- The textbook *Applied Deep Learning* (Fergus & Chalmers), Chapter 7 — object detection.
- Lecture 6: *From classification to detection — bounding boxes, IoU, and the YOLO family of models*.

## A different kind of lab

Up to now every lab has been about *training a model*. This one is about **preparing the data a model will train on**. That sounds less glamorous than the modelling work, but in industry the ratio is typically 70/30 data work to modelling work, sometimes much higher. **The quality ceiling on any deep learning project is set by the quality of the labels.** A model trained on inconsistent or careless annotations will be inconsistent or careless. There is no clever architecture that fixes bad data.

Today we annotate. Next week we'll train an object detector on the dataset you produce here.

## What you'll build

By the end of this session you will have:

- A folder of 15 pre-supplied UK wildlife images, each annotated with bounding boxes for any animals visible from the five-class list: **badger, fox, deer, hedgehog, squirrel**.
- A matching folder of YOLO-format `.txt` files, one per image.
- A `data.yaml` declaration file that the `ultralytics` training pipeline can ingest.
- A train/val/test split appropriate for object detection.

## Useful references

- [Ultralytics dataset format documentation](https://docs.ultralytics.com/datasets/detect/)
- [Roboflow's guide to writing good bounding boxes](https://blog.roboflow.com/labeling/) — vendor blog but covers the hygiene rules well.

---

## 1. The YOLO text format, dissected

Every object detector needs to know *where* objects are, not just *what* they are. The YOLO family encodes a bounding box as four numbers, plus a class id, all on one line:

```
class_id  cx  cy  w  h
```

For example:

```
1 0.523000 0.487000 0.214000 0.299000
```

Reading this off:

| Field      | Meaning |
|------------|---------|
| `1`        | Class id (in our dataset: `0=badger, 1=fox, 2=deer, 3=hedgehog, 4=squirrel`) |
| `0.523000` | **`cx`** — x-coordinate of box centre, **normalised** (0 = left edge, 1 = right edge) |
| `0.487000` | **`cy`** — y-coordinate of box centre, **normalised** (0 = top, 1 = bottom) |
| `0.214000` | **`w`** — box width, **normalised** (fraction of image width) |
| `0.299000` | **`h`** — box height, **normalised** (fraction of image height) |

**Two things worth pausing on:**

1. The format uses the box **centre**, not the corner. Many other formats (Pascal VOC, COCO) use `xmin, ymin, xmax, ymax` corners. Mixing them up is a classic bug — see Exercise 2.
2. All four geometric fields are **normalised by image dimensions**. This means the same `.txt` file is valid whether the image is 800×600 or 4096×3072 — the label is intrinsically resolution-independent. This is why YOLO's format scales so well across heterogeneous datasets.

**Per-image rule:** one `.txt` file per image, named with the same stem (e.g. `fox_01.jpg` → `fox_01.txt`). One row per object instance. If there are no objects in the image, the `.txt` file is empty (but should still exist).

## 2. The hands-on bit — open the annotator

The module's container ships with a built-in bounding-box annotator that writes YOLO-format files directly into the bind-mounted `data/labels/` folder. **Open it now in a new tab:**

> **▶ <http://localhost:8000/annotator>**

(Or click the *Open annotator* button on the launcher home page.)

### What you should see

Three panels: a list of all 15 images on the left, a canvas with the current image in the middle, and a class picker plus a live view of the YOLO label file on the right. The keyboard shortcuts at the bottom of the canvas — click-and-drag to draw, number keys to pick a class, arrow keys to move between images — are the fastest way to work.

**Zooming in for small objects.** Some of the starter images contain animals that are very small relative to the frame (e.g. `small_fox.jpg`, `tiny_hedgehog.jpg`). Drawing a tight box around something tiny is fiddly at default zoom. Use the **mouse wheel** to zoom in toward the cursor (up to 12×), and **Shift + click-drag** to pan around once you're zoomed in. The `+` / `-` / `0` keys also work for zoom in / out / fit-to-pane. There are zoom buttons in the top-right corner of the canvas if you forget the shortcuts. **Crucially: the saved coordinates are unaffected by zoom level** — zoom is just a viewing aid, the YOLO `.txt` file always contains normalised `[0, 1]` values regardless of how you drew the boxes.

**For small or hard-to-see objects**, use the **zoom and pan** controls in the top-right corner of the canvas. Mouse-wheel zooms in and out *at the cursor* (try it — it's worth getting fluent with). Hold **Shift and drag** to pan when zoomed in. Press **`0`** or click *Fit* to reset to the whole image. Tight boxes on a small distant fox are much easier at 4× zoom than at fit-to-screen.

**Zoom and pan.** Some images contain small or distant objects — `tiny_hedgehog.jpg` is a good example, where the animal occupies less than 3% of the image area. You can zoom in for these:

- **Scroll wheel** to zoom toward your cursor (the image pixel under the cursor stays under the cursor — like Google Maps).
- **`+` and `-`** keys to zoom from the canvas centre.
- **`0`** key or the **Fit** button in the top-right zoom panel to reset the view.
- **Shift + click-drag** to pan when zoomed in.

Zoom is purely a view tool — it never changes the coordinates stored in your `.txt` file. Watch the YOLO panel on the right while you draw at high zoom: the normalised values are still in `[0, 1]` because they're based on the underlying image, not the on-screen view.

### Your task

**Annotate all 15 images.** For each one:

1. Look at the image. Identify every animal that's clearly visible.
2. Pick the right class (1–5 keys or the buttons).
3. Draw a tight box around the animal — including everything that's obviously part of it (tail, legs, ears), excluding background, *not* including parts of other animals.
4. If the animal is **partly cut off by the edge of the image**, still draw a box around the visible part. Truncation is normal in real datasets and detectors handle it fine.
5. If there are **multiple animals**, draw one box per animal.
6. If you're **unsure of the species**, skip it for now. Better an unlabelled example than a wrong one. (We'll return to this in Exercise 2.)

The annotator auto-saves on every change. There's no save button. As you draw, watch the *YOLO label file* panel on the right — you're literally writing the rows of the label file as you click and drag.

**Aim for tight boxes.** Loose boxes leak background pixels into the model's idea of what 'fox' looks like, and the detector learns that 'fox = lots of grass with a smaller fox-shape in it'. Tight is better.

**Come back here once you've labelled all 15 images.** The 'image list' panel on the left will show how many boxes each image has. Aim for 'all 15 labelled, with a total of around 18-20 boxes across the set' (because some images have multiple animals).

## 3. Inspect your work — load the labels back into Python

**Always validate your annotations visually.** Reading the raw `.txt` numbers tells you nothing about whether the boxes are sensible. Re-rendering them on top of the images closes the loop and catches:

- Off-by-one errors (e.g. confused row/column order)
- Class-id mistakes ("this is labelled `2 deer` but actually it's a fox")
- Boxes that drifted from the object during a resize
- Truncation issues at image edges

We use the same `PIL` + `matplotlib` we've been using all module. No special libraries needed.

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import random

DATA_DIR = Path("data")
IMAGES_DIR = DATA_DIR / "images"
LABELS_DIR = DATA_DIR / "labels"

# Read the class list — single source of truth.
with open(DATA_DIR / "classes.txt") as f:
    classes = [line.strip() for line in f if line.strip()]
print(f"Classes ({len(classes)}): {classes}")

# A per-class colour list for plotting — matches the annotator's palette.
CLASS_COLOURS = {
    0: "#d6336c",   # badger
    1: "#f59f00",   # fox
    2: "#2b8a3e",   # deer
    3: "#9a59f4",   # hedgehog
    4: "#1c7ed6",   # squirrel
}

In [ ]:
def read_yolo_labels(label_path: Path) -> list[dict]:
    """Parse a YOLO .txt file. Returns a list of dicts with class_id, cx, cy, w, h."""
    if not label_path.is_file():
        return []
    boxes = []
    for line in label_path.read_text().splitlines():
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        if len(parts) != 5:
            print(f"  WARNING: malformed line in {label_path.name}: {line!r}")
            continue
        cls_id = int(parts[0])
        cx, cy, w, h = [float(x) for x in parts[1:]]
        boxes.append({"class_id": cls_id, "cx": cx, "cy": cy, "w": w, "h": h})
    return boxes


def draw_image_with_boxes(image_path: Path, boxes: list[dict], ax=None):
    """Render an image with overlaid bounding boxes on the given axis."""
    img = Image.open(image_path)
    W, H = img.size
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(img)
    ax.axis("off")

    for b in boxes:
        # Convert centre-based normalised back to corner-based pixels.
        x = (b["cx"] - b["w"] / 2) * W
        y = (b["cy"] - b["h"] / 2) * H
        w = b["w"] * W
        h = b["h"] * H
        colour = CLASS_COLOURS.get(b["class_id"], "#888888")
        rect = patches.Rectangle((x, y), w, h, linewidth=2.5,
                                  edgecolor=colour, facecolor="none")
        ax.add_patch(rect)
        # Label background and text
        label = classes[b["class_id"]] if 0 <= b["class_id"] < len(classes) else f"cls{b['class_id']}"
        ax.text(x + 4, y - 6, label, color="white", fontsize=11, fontweight="bold",
                bbox=dict(boxstyle="square,pad=0.2", facecolor=colour, edgecolor="none"))
    return ax

Now look at every image with its current annotations. **If you haven't annotated yet, this will show blank images** — open the annotator and label them first.

In [ ]:
# Walk the images folder in sorted order so the output is deterministic.
image_files = sorted(IMAGES_DIR.glob("*.jpg"))

# Cap how many we render. Matplotlib has a 2^16 (=65,536) pixel limit per axis,
# and at 4 inches per row a large dataset blows past that. The bundled set is
# only 15 images so this kicks in only if you add your own data later.
MAX_IMAGES_TO_SHOW = 24
to_show = image_files[:MAX_IMAGES_TO_SHOW]
n_imgs = len(to_show)
n_cols = 3
n_rows = (n_imgs + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = axes.flatten() if n_rows > 1 else [axes] if n_cols == 1 else axes

# Count boxes across the WHOLE dataset, even though we only render some.
total_boxes = 0
for img_path in image_files:
    label_path = LABELS_DIR / (img_path.stem + ".txt")
    total_boxes += len(read_yolo_labels(label_path))

# Render only the capped subset.
for ax, img_path in zip(axes, to_show):
    label_path = LABELS_DIR / (img_path.stem + ".txt")
    boxes = read_yolo_labels(label_path)
    draw_image_with_boxes(img_path, boxes, ax=ax)
    ax.set_title(f"{img_path.name}  ({len(boxes)} box{'es' if len(boxes) != 1 else ''})", fontsize=10)

# Hide any leftover axes
for ax in axes[n_imgs:]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

if len(image_files) > MAX_IMAGES_TO_SHOW:
    print(f"\nShowing first {MAX_IMAGES_TO_SHOW} of {len(image_files)} images.")
print(f"Total: {total_boxes} boxes across {len(image_files)} images.")

**What to look for:**

- Every animal in every image has a box.
- Every box is tight around its animal (no big gaps of background inside the box).
- Every box has the right class — colour-check it against the species you can see.
- Boxes that look 'shifted' from their animal usually mean an off-by-one bug in the annotator. We've tested ours but if you see something weird, report it.

**Re-open the annotator and fix anything you spot.** The label files update live; rerun the cell above to see the latest.

## 4. The `data.yaml` file

`ultralytics` (the library that ships YOLOv8/v11/26) needs one extra file: a `data.yaml` that tells it the dataset layout. It's a small file but it's the contract between your annotations and the trainer.

Required fields:

```yaml
path: /workspace/labs/lab06_image_annotation/data    # absolute path to the dataset root
train: images/train                                    # train images, relative to `path`
val:   images/val                                      # validation images, relative to `path`
test:  images/test                                     # test images (optional but recommended)

names:
  0: badger
  1: fox
  2: deer
  3: hedgehog
  4: squirrel
```

`ultralytics` looks for label files in the *parallel* folder `labels/<split>/` next to each `images/<split>/`. So our folder structure needs to be reorganised slightly from the flat one we used for annotation.

## 5. Train / val / test split for object detection

We split into 70% train / 15% val / 15% test, but for object detection this needs a slightly different mindset from classification.

**Important: avoid frame-level leakage.** If your images came from a video — say, three frames at 0.04s intervals showing the same fox in nearly the same pose — random shuffling will put one frame in train and another in val, and the validation score will be massively optimistic. The model will essentially have seen the validation image during training.

For our 15 independent synthetic images this isn't a risk. For *your own* dataset later, **split at the level of unique scenes**, not individual frames.

In [ ]:
import shutil
import yaml
import random

RNG_SEED = 7144
SPLIT = (0.7, 0.15, 0.15)

# Find every image that has at least one annotation. Images without
# annotations are excluded from training — you wouldn't want to teach
# the model 'this image is correctly labelled as containing nothing'
# unless you really do want negative examples.
labelled = [p for p in sorted(IMAGES_DIR.glob("*.jpg"))
            if (LABELS_DIR / (p.stem + ".txt")).is_file()
            and read_yolo_labels(LABELS_DIR / (p.stem + ".txt"))]

print(f"Labelled images available: {len(labelled)} of {len(image_files)}")

if len(labelled) < len(image_files):
    print("You haven't annotated every image yet — go back to step 2.")
else:
    rng = random.Random(RNG_SEED)
    rng.shuffle(labelled)
    n = len(labelled)
    n_train = int(n * SPLIT[0])
    n_val = int(n * SPLIT[1])
    splits = {
        "train": labelled[:n_train],
        "val":   labelled[n_train:n_train + n_val],
        "test":  labelled[n_train + n_val:],
    }
    for split_name, files in splits.items():
        print(f"  {split_name:<5}: {len(files)} images")

In [ ]:
# Build the directory layout that ultralytics expects.
# We create images/{train,val,test} and labels/{train,val,test} side-by-side.

OUT_ROOT = DATA_DIR  # we write the split structure right next to the flat one

for split in ("train", "val", "test"):
    (OUT_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUT_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)

# We *copy* (not move) so the flat data/images/ stays intact for re-annotation.
for split_name, files in splits.items():
    for fp in files:
        shutil.copy2(fp, OUT_ROOT / "images" / split_name / fp.name)
        label_src = LABELS_DIR / (fp.stem + ".txt")
        shutil.copy2(label_src, OUT_ROOT / "labels" / split_name / (fp.stem + ".txt"))
    print(f"  {split_name}: {len(files)} files copied to images/{split_name}/ and labels/{split_name}/")

# Write the data.yaml. We use a relative `path` so the file works regardless
# of where the container is mounted on the host.
data_yaml = {
    "path": str((OUT_ROOT).resolve()),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {i: name for i, name in enumerate(classes)},
}

yaml_path = OUT_ROOT / "data.yaml"
with open(yaml_path, "w") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

print(f"\nWrote {yaml_path}")
print("\nContents:")
print(yaml_path.read_text())

**Sanity check.** When we get to the YOLO training lab, the trainer will fail with a useful error if anything is wrong with this setup. But let's catch problems now while we still know what we did. Three checks:

1. Every image in `images/<split>/` has a corresponding `labels/<split>/<stem>.txt`.
2. Every class id in every label file is within range `0..len(classes)-1`.
3. Every box's coordinates are within `[0, 1]`.

In [ ]:
issues = []

for split in ("train", "val", "test"):
    img_dir = OUT_ROOT / "images" / split
    lbl_dir = OUT_ROOT / "labels" / split
    for img_path in sorted(img_dir.glob("*.jpg")):
        lbl_path = lbl_dir / (img_path.stem + ".txt")
        if not lbl_path.is_file():
            issues.append(f"{split}: no label file for {img_path.name}")
            continue
        for i, line in enumerate(lbl_path.read_text().splitlines(), start=1):
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) != 5:
                issues.append(f"{split}/{lbl_path.name} line {i}: not 5 fields")
                continue
            try:
                cls_id = int(parts[0])
                cx, cy, w, h = [float(x) for x in parts[1:]]
            except ValueError:
                issues.append(f"{split}/{lbl_path.name} line {i}: parse error")
                continue
            if not (0 <= cls_id < len(classes)):
                issues.append(f"{split}/{lbl_path.name} line {i}: class_id {cls_id} out of range")
            for fname, v in [("cx", cx), ("cy", cy), ("w", w), ("h", h)]:
                if not (0 <= v <= 1):
                    issues.append(f"{split}/{lbl_path.name} line {i}: {fname}={v} outside [0,1]")

if issues:
    print(f"{len(issues)} issue(s) found:")
    for issue in issues:
        print(f"  - {issue}")
else:
    print("All checks passed. Your dataset is ready to train on.")

---

## 6. Exercise 1 — annotate your own images

The 15 stylised images you've just labelled are deliberately easy — the animals are clearly visible against simple backgrounds. Real wildlife photos are far harder, and that's where the annotation craft becomes visible.

**Your task:**

1. Source **at least 5 real photos** of UK wildlife from any of: [iNaturalist](https://www.inaturalist.org/observations) (filter for CC-licensed), [Wikimedia Commons](https://commons.wikimedia.org/), your own phone, your university's Conservation AI portal, or any other source you have rights to.
2. Drop the JPEGs into `data/images/` (you can do this from your host machine — the folder is bind-mounted).
3. Refresh the annotator at <http://localhost:8000/annotator> — your new images should appear in the left-hand list.
4. Annotate them with the same five classes as before. **For each photo, note in writing one thing that was harder than the synthetic set.**
5. Re-run the verification cells above (3 and the validation cell) on the extended dataset.

Write your reflections in the markdown cell below this code cell.

In [ ]:
# Use this cell to re-run verification on your extended set if you like.


*Your written observations on annotating real photos (vs. the synthetic set):*



## 7. Exercise 2 — annotation hygiene & edge cases

Real annotation is full of judgement calls. For **each** of the scenarios below, write **2–4 sentences** describing how you would handle it. There isn't always a single right answer — you're being assessed on whether your judgement is **defensible and consistent**.

**(a) Truncation.** A fox is half-hidden behind a tree, with only its head and shoulders visible. Do you draw a box, and if so, around what — only the visible part, or your best guess of the whole animal's outline?

**(b) Multiple instances stacked.** Three squirrels are huddled together in a tight group, partially overlapping. Do you draw three separate boxes (one per animal), or one big box?

**(c) Ambiguous species.** A small mammal in low light could plausibly be a badger or a hedgehog. What do you do?

**(d) Off-class object.** Your photo prominently features a rabbit. Rabbits are not in your five-class list. What do you do with it?

**(e) Cropping vs. labelling.** A very out-of-focus animal in the corner of the image — barely identifiable as anything. Annotate or skip?

**(f) Inter-annotator consistency.** You're working with three other students on the same dataset. How do you ensure you're all drawing boxes the same way?

*Your answers:*

**(a) Truncation:** 

**(b) Multiple instances stacked:** 

**(c) Ambiguous species:** 

**(d) Off-class object:** 

**(e) Out-of-focus tiny object:** 

**(f) Inter-annotator consistency:** 

## 8. Exercise 3 — inter-annotator agreement

**Pair up with one classmate.** Pick **the same single image** from the 15 starter images, and re-annotate it independently — neither of you should see the other's boxes while drawing.

Once you've both finished:

1. Compare your two `.txt` files side-by-side.
2. For each box in your file, find the 'matching' box in your partner's file (the one closest in centre and size).
3. For each pair, compute the **Intersection over Union (IoU)** — the area of overlap divided by the area of union. The function below computes IoU; use it.
4. Report the mean IoU across all matched pairs.
5. If there are 'extra' boxes in one annotator's file that don't appear in the other's, discuss why.

**Discussion.** A mean IoU of 0.9 across annotators is excellent. 0.7 is good. Below 0.5 means you're drawing meaningfully different boxes and you have a *calibration problem* — write one sentence describing what calibration step you'd add to your team's process.

In [ ]:
def iou(box1: dict, box2: dict) -> float:
    """Intersection over Union for two YOLO-format boxes.

    Both boxes are dicts with cx, cy, w, h in normalised [0, 1] coordinates.
    Returns a value between 0 (no overlap) and 1 (perfect overlap).
    """
    # Convert centre-based to corner-based.
    def corners(b):
        return (b["cx"] - b["w"] / 2, b["cy"] - b["h"] / 2,
                b["cx"] + b["w"] / 2, b["cy"] + b["h"] / 2)
    x1a, y1a, x2a, y2a = corners(box1)
    x1b, y1b, x2b, y2b = corners(box2)
    # Intersection rectangle
    inter_w = max(0.0, min(x2a, x2b) - max(x1a, x1b))
    inter_h = max(0.0, min(y2a, y2b) - max(y1a, y1b))
    inter = inter_w * inter_h
    if inter == 0:
        return 0.0
    union = box1["w"] * box1["h"] + box2["w"] * box2["h"] - inter
    return inter / union


# Quick self-test: identical box -> IoU = 1
b1 = {"cx": 0.5, "cy": 0.5, "w": 0.4, "h": 0.4}
b2 = {"cx": 0.5, "cy": 0.5, "w": 0.4, "h": 0.4}
print(f"identical:  IoU = {iou(b1, b2):.3f}")
# Shifted by 10% of image -> still good
b3 = {"cx": 0.55, "cy": 0.55, "w": 0.4, "h": 0.4}
print(f"shifted:    IoU = {iou(b1, b3):.3f}")
# Non-overlapping -> 0
b4 = {"cx": 0.9, "cy": 0.9, "w": 0.1, "h": 0.1}
print(f"disjoint:   IoU = {iou(b1, b4):.3f}")

In [ ]:
# Your code for Exercise 3 — comparing your boxes against your partner's.


*Your written reflection on inter-annotator agreement:*

**Mean IoU achieved:** 

**Disagreements:** 

**Calibration step you'd add:** 

---

## 9. Reflection questions

**Q1.** State, in one sentence, what makes object detection a fundamentally harder problem than image classification.

**Q2.** Why does the YOLO label format use **normalised** coordinates (in `[0, 1]`) rather than absolute pixel coordinates? Give one practical benefit and one practical drawback.

**Q3.** The lab earlier insisted that empty label files (for images containing no annotated objects) should *still exist* on disk. What would go wrong if we just left such images without `.txt` files at all?

**Q4.** In Section 5 we warned about 'frame-level leakage' when splitting object-detection datasets. Give a specific example of a dataset structure (real or hypothetical) where naively shuffling by image would create this leakage, and describe how you'd fix it.

**Q5.** Annotation is expensive — the rule of thumb in industry is roughly 30 seconds per box for trained annotators. Suppose your team has one week and £5,000 of budget and you need a dataset of 5,000 wildlife images annotated for the five classes used in this lab. Outline a plan: do you do it in-house, outsource, use semi-automated tools, or some combination? Justify your choice in 3–5 sentences.

*Your answers:*

**A1.** 

**A2.** 

**A3.** 

**A4.** 

**A5.** 

---

## What's next

You now have a clean, annotated, train/val/test-split YOLO dataset on disk, with a valid `data.yaml`. **In Lab 7 we will train an object detection model on it** — using a pre-trained YOLO model and the Ultralytics library — and finally use everything we've built in the past six weeks to find UK wildlife in unseen images.

Before leaving today, make sure:

- [ ] All 15 starter images have at least one bounding box (where appropriate)
- [ ] You have completed Exercises 1, 2, and 3
- [ ] You have answered the reflection questions
- [ ] The validation cell in Section 5 reports zero issues
- [ ] `data/data.yaml` exists and has been created with the correct paths
- [ ] Your notebook runs **top to bottom without errors** (*Kernel → Restart and Run All*)